# Step 1 check on Kaggle

Confirms that the data is found, logging works, and training resumes exactly after a crash.
The network here is throwaway; it is not a baseline.

**To save GPU quota, run this twice:**
1. Accelerator **None** (CPU). This costs no GPU hours and catches most problems.
2. Accelerator **GPU T4 x2**, only after the CPU run passes.

Before running: attach your Kaggle Dataset, turn **Internet on**, and set `BRANCH` below.
If the repository is private, add a Kaggle Secret named `GITHUB_TOKEN`.


In [ ]:
import shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
!PYTHONPATH=src python -m pytest tests/test_exp_setup.py -q


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.smoke --config configs/smoke.yaml


In [ ]:
import json

runs = sorted(Path("/kaggle/working/runs").glob("*_smoke_s*"))
summary = json.loads((runs[-1] / "summary.json").read_text())
env = json.loads((runs[-1] / "env.json").read_text())
print("PASSED" if summary["passed"] else "FAILED")
print(json.dumps(summary, indent=2))
print(json.dumps({k: env[k] for k in ("python", "numpy", "pandas", "opencv", "torch", "cuda", "gpus", "git_commit", "git_dirty", "kaggle", "account")}, indent=2))
